In [1]:
from pathlib import Path
from copy import deepcopy

import matplotlib.pyplot as plt

from contact_plots import get_data_from_files, create_and_save_box_paper_plot, create_base_case_paper_plot

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder, convert_2_minutes=True)


In [3]:
cut_off_distances = ['1.0', '1.5', '2.0']

data_names_base = (
    'customer_customer_weight_over_contacts',
    'customer_customer_weight_over_agents',
    'customer_customer_contacts_over_agents',
    'staff_customer_weight_over_contacts',
    'staff_customer_weight_over_agents',
    'staff_customer_contacts_over_agents',
    'customer_staff_weight_over_agents',
    'customer_staff_contacts_over_agents',
)

parts_2_data_name = {}
for cut_off_distance in cut_off_distances:
    data_name_dict = {}
    for data_name in data_names_base:
        parts = data_name.split('_')
        interaction_type = '_'.join(parts[:2])
        data_type = '_'.join(parts[2:])
        if interaction_type not in data_name_dict:
            data_name_dict[interaction_type] = {}
            
        data_name_dict[interaction_type][data_type] = f'{cut_off_distance}_{data_name}'
    
    parts_2_data_name[cut_off_distance] = data_name_dict
    
measures_and_labels = (
    ('nn_nts_bvd', 'Base'),
    ('n_nts_bvd', 'Neighbourhoods'),
    ('nn_ts_bvd', 'Time slots'),
    ('nn_nts_svd', 'Shorter visits'),
    ('nn_nts_lvd', 'Longer visits')
)

base_measure_name = 'nn_nts_bvd'

In [ ]:
divider_config = {
    'left':0.65,  
    'gap_hor':0.65,
    'right':0.1,
    'bottom':1.00,
    'gap_ver':0,
    'top':0.30,
}

measure_names = [measure_name for measure_name, _ in measures_and_labels]
labels = [label for _, label in measures_and_labels]

flier_props = {'markersize':6, 'marker':'x', 'markeredgecolor':'0.2',
                  'linewidth':1, 'markeredgewidth':1}
    

box_plot_config = {'vert':True, 'notch':True, 'labels':labels, 'patch_artist':True, 'widths':0.8, 'flierprops':flier_props,
                   'whiskerprops':{'linewidth':1},'capprops':{'linewidth':2},
                    'boxprops':{'linewidth':1, 'fill':False}, 'medianprops':{'linewidth':1, 'color':'r'}}#, 'facecolor':'tab:blue'}}

violin_plot_config = {'widths':0.8, 'showmeans':False, 'showextrema':False, 'showmedians':False}
violin_body_plot_config = {'edge_color':'k', 'face_color':'tab:blue', 'alpha':0.8}


bar_plot_config = {'width':0.8}

bar_plot_legend_config = {'loc':'upper center', 'ncol':3, 'title':'Contact counts', 'fontsize':9, 'handlelength':1.2, 
                          'handleheight':0.5, 'title_fontsize':9, 'columnspacing':1.0, 'handletextpad':0.6, 'bbox_to_anchor':(0.5, 1.02)}
bar_plot_text_config = {'fontsize':8, 'horizontalalignment':'center', 'verticalalignment':'center'}

main_ax_config = {'grid':{'visible':True, 'axis':'y'}, 'y_label_config':{'fontsize':9},
                 'x_tick_labels':labels, 'x_tick_labels_config':{'fontsize':9, 'rotation':45, 'ha':'right'},
                 'y_tick_labels_config':{'fontsize':9}, 'title_config':{'fontsize':10}}

mean_bar_config = {'width':0.8, 'edgecolor':'k', 'linewidth':0.5}
mean_bar_color_config = {'base':'tab:blue', 'positive':'tab:green', 'negative':'tab:red'}
#top_ax_config = {'grid':{'visible':True}, 'x_tick_labels':[], 'y_label':'Rel. diff.\n Mean value\n [%]',
#                 'y_label_config':{'fontsize':10}, 'y_lim':[-100,100], 'title_config':{'fontsize':12}}

thresholds_config = [{'value':5, 'line_config':{'color':'tab:blue', 'linestyle':'-'}},
                    {'value':10, 'line_config':{'color':'tab:red', 'linestyle':'--'}},
                    {'value':15, 'line_config':{'color':'tab:green', 'linestyle':':'}}]

v_line_config = {'linewidth':1.5, 'color':'0.6', 'linestyle':'--', 'zorder':2}

base_plot_config = {'main_type':'box_plot', 'measure_names':measure_names, 'box_plot_config':box_plot_config,
                   'main_ax_config':main_ax_config, 'mean_bar_config':mean_bar_config,
                   'v_line_config':v_line_config, 'mean_bar_color_config':mean_bar_color_config, 
                    'bar_plot_config':bar_plot_config, 'bar_plot_legend_config':bar_plot_legend_config,
                   'bar_plot_text_config':bar_plot_text_config, 'thresholds_config':thresholds_config,
                   'violin_plot_config':violin_plot_config, 'violin_body_plot_config':violin_body_plot_config}

plot_types_to_y_labels_and_titles = (
    ('weight_over_contacts', 'Weight [min]', 'Weight over contacts (a)'),
    ('weight_over_agents', 'Weight [min]', 'Weight over agents (b)'),
    ('contacts_over_agents', 'Count [-]', 'Contacts over agent (c)')
)

interaction_type_labels = {'customer_customer':'Customer - Customer', 
                           'customer_staff':'Customer - Staff', 
                           'staff_customer':'Staff - Customer'}

fig_config = {'figsize':(6.5, 4)}

save_pd = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Graphics')


In [5]:
%matplotlib wx

#plt.style.use('ggplot')

infra_name_all = 'base_case'
interaction_type_all = 'customer_customer'
cut_off_dist = '1.5'

data_name_slice = parts_2_data_name[cut_off_dist][interaction_type_all]

save_filename = save_pd.joinpath(f'{infra_name_all}_{interaction_type_all}_box.pdf')

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for data_type, y_label, title in plot_types_to_y_labels_and_titles:
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub.update({'data_name':data_name_slice[data_type]})
    plot_config_sub['main_ax_config'].update({'y_label':y_label, 'title':title})
    plots_config.append(plot_config_sub)
    
create_and_save_box_paper_plot(fig_config, save_filename, divider_config, plots_config, infra_and_measure_name_2_data)

In [6]:
%matplotlib wx

infra_name_all = 'base_case'
interaction_type_all = 'staff_customer'
cut_off_dist = '1.5'

data_name_slice = parts_2_data_name[cut_off_dist][interaction_type_all]

save_filename = save_pd.joinpath(f'{infra_name_all}_{interaction_type_all}_box.pdf')

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for data_type, y_label, title in plot_types_to_y_labels_and_titles:
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub.update({'data_name':data_name_slice[data_type]})
    plot_config_sub['main_ax_config'].update({'y_label':y_label, 'title':title})
    plots_config.append(plot_config_sub)
    
create_and_save_box_paper_plot(fig_config, save_filename, divider_config, plots_config, infra_and_measure_name_2_data)

In [7]:
%matplotlib wx

infra_name_all = 'base_case'
interaction_type_all = 'customer_staff'
cut_off_dist = '1.5'

data_name_slice = parts_2_data_name[cut_off_dist][interaction_type_all]

save_filename = save_pd.joinpath(f'{infra_name_all}_{interaction_type_all}_box.pdf')

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for data_type, y_label, title in plot_types_to_y_labels_and_titles:
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_ax_config'].update({'y_label':y_label, 'title':title})
    
    if data_type == 'weight_over_contacts':
        plot_config_sub.update({'data_name':parts_2_data_name[cut_off_dist]['staff_customer'][data_type]})
    else:
        plot_config_sub.update({'data_name':data_name_slice[data_type]})
    if data_type == 'contacts_over_agents':
        plot_config_sub['main_type'] = 'bar_plot'
        plot_config_sub['main_ax_config']['y_lim'] = [0,1.2]
        plot_config_sub['main_ax_config']['y_label'] = 'Distribution of counts [-]'
    
    plots_config.append(plot_config_sub)
    
create_and_save_box_paper_plot(fig_config, save_filename, divider_config, plots_config, infra_and_measure_name_2_data)

In [8]:
#plt.show()

In [6]:
%matplotlib wx

infra_name_all = 'base_case'
data_type_all = 'weight_over_contacts'
y_label_all =  'Weight [min]'
cut_off_dist = '1.5'

save_filename = save_pd.joinpath(f'{infra_name_all}_{data_type_all}_violin.pdf')

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for ii, interaction_type in enumerate(('customer_customer', 'staff_customer')):
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_type'] = 'violin_plot'
    plot_config_sub.update({'data_name':parts_2_data_name[cut_off_dist][interaction_type][data_type_all]})
    plot_config_sub['main_ax_config'].update({'y_label':y_label_all, 'title':f'({chr(97+ii)}) {interaction_type_labels[interaction_type]}'})
    plots_config.append(plot_config_sub)
    
create_and_save_box_paper_plot(fig_config, save_filename, divider_config, plots_config, infra_and_measure_name_2_data)

In [7]:
%matplotlib wx

infra_name_all = 'base_case'
data_type_all = 'weight_over_agents'
y_label_all =  'Weight [min]'
cut_off_dist = '1.5'

save_filename = save_pd.joinpath(f'{infra_name_all}_{data_type_all}_violin.pdf')

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for ii, interaction_type in enumerate(interaction_type_labels):
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_type'] = 'violin_plot'
    plot_config_sub.update({'data_name':parts_2_data_name[cut_off_dist][interaction_type][data_type_all]})
    plot_config_sub['main_ax_config'].update({'y_label':y_label_all, 'title':f'({chr(97+ii)}) {interaction_type_labels[interaction_type]}'})
    plots_config.append(plot_config_sub)
    
create_and_save_box_paper_plot(fig_config, save_filename, divider_config, plots_config, infra_and_measure_name_2_data)

In [10]:
%matplotlib wx

infra_name_all = 'base_case'
data_type_all = 'contacts_over_agents'
y_label_all =  'Count [-]'
cut_off_dist = '1.5'

save_filename = save_pd.joinpath(f'{infra_name_all}_{data_type_all}_violin.pdf')

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for ii, interaction_type in enumerate(interaction_type_labels):
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_type'] = 'violin_plot'
    plots_config.append(plot_config_sub)
    plot_config_sub.update({'data_name':parts_2_data_name[cut_off_dist][interaction_type][data_type_all]})
    plot_config_sub['main_ax_config'].update({'y_label':y_label_all, 'title':f'({chr(97+ii)}) {interaction_type_labels[interaction_type]}'})
    if interaction_type == 'customer_staff':
        plot_config_sub['main_type'] = 'bar_plot'
        plot_config_sub['main_ax_config']['y_lim'] = [0,1.2]
        plot_config_sub['main_ax_config']['y_label'] = 'Distribution of counts [-]'
    
create_and_save_box_paper_plot(fig_config, save_filename, divider_config, plots_config, infra_and_measure_name_2_data)

AttributeError: 'Rectangle' object has no property 'widths'

In [5]:
%matplotlib wx

infra_name_all = 'base_case'
interaction_type_all = 'staff_customer'
cut_off_dist = '1.5'

data_name_slice = parts_2_data_name[cut_off_dist][interaction_type_all]

divider_config['top_axes_scale'] = 0.2

plot_config = deepcopy(base_plot_config)
plot_config.update({'infra_name':infra_name_all})
plots_config = []

for data_type, y_label, title in plot_types_to_y_labels_and_titles:
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_type'] = 'cdf_plot'
    plot_config_sub.update({'data_name':data_name_slice[data_type]})
    plot_config_sub['main_ax_config'].update({'x_label':y_label, 'title':title})
    plot_config_sub['main_ax_config'].pop('x_tick_labels')
    plots_config.append(plot_config_sub)
    
create_base_case_paper_plot(fig_config, divider_config, plots_config, infra_and_measure_name_2_data)

plt.show()

No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
